# DAG Metric — Deterministic Decision Trees for Evaluation

The **DAG (Directed Acyclic Graph) Metric** lets you build structured, deterministic evaluation trees that combine binary/non-binary judgement nodes with optional GEval scoring.

### Why DAGs?
- **Transparent**: Every decision path is visible and debuggable
- **Deterministic**: Same input always follows the same decision tree
- **Composable**: Mix rule-based checks with LLM judgements
- **Hybrid**: Leaf nodes can delegate to GEval for subjective scoring

### Prerequisites
- Notebook 05 (familiarity with G-Eval)

---
## 1. Setup & Imports

In [ ]:
import os
import json
from dotenv import load_dotenv

dotenv_path = os.path.join(os.path.dirname(os.getcwd()), ".env")
load_dotenv(dotenv_path)

import numpy as np
import pandas as pd
from deepeval.test_case import LLMTestCase, LLMTestCaseParams
from deepeval.metrics import GEval

print("Setup complete.")

---
## 2. Load Test Data

In [ ]:
# Load pipeline results from Notebook 02
data_path = os.path.join(os.getcwd(), "data", "pipeline_results.json")

with open(data_path) as f:
    pipeline_results = json.load(f)

# Create test cases
test_cases = []
for r in pipeline_results:
    tc = LLMTestCase(
        input=r["query"],
        actual_output=r["actual_output"],
        expected_output=r.get("expected_output", ""),
        retrieval_context=r["retrieval_context"],
    )
    test_cases.append(tc)

print(f"Loaded {len(test_cases)} test cases.")

-
## DAG Metric - Deterministic Decision Trees for LLM Evaluation

The **DAG (Directed Acyclic Graph) Metric** provides deterministic control over LLM evaluation by letting you define explicit decision trees with scored outcomes. Unlike G-Eval, which gives the LLM freedom to score, DAG forces evaluation through pre-defined paths with fixed scores.

### When to Use DAG vs GEval

| Aspect | GEval | DAG |
|---|---|---|
| Control | LLM decides score freely | You define exact scoring paths |
| Determinism | Non-deterministic | Deterministic decision tree |
| Best for | Subjective criteria (tone, clarity) | Objective criteria (format, structure) |
| Complexity | Simple to define | Requires tree design |
| Flexibility | High | Medium (bounded by your tree) |

### DAG Node Types

1. **TaskNode** - Processes/extracts data from test case parameters. Think of it as a preprocessing step.
2. **BinaryJudgementNode** - Yes/No decision point. Has exactly 2 VerdictNode children (True/False).
3. **NonBinaryJudgementNode** - Multiple-choice decision point. Has N VerdictNode children for different outcomes.
4. **VerdictNode** - Leaf node that assigns a final score (0-10) or delegates to a child node/metric for further evaluation.

In [ ]:
from deepeval.metrics import DAGMetric
from deepeval.metrics.dag import (
    DeepAcyclicGraph,
    TaskNode,
    BinaryJudgementNode,
    NonBinaryJudgementNode,
    VerdictNode,
)

print("DAG imports successful.")
print("Available node types: TaskNode, BinaryJudgementNode, NonBinaryJudgementNode, VerdictNode")

### DAG Example 1: Format Correctness (from DeepEval Docs)

This is the canonical example from the DeepEval documentation. It evaluates whether a meeting summary has the correct headings (intro, body, conclusion) in the right order.

**Decision tree:**
```
TaskNode: Extract headings from actual_output
    +-- BinaryJudgementNode: Has all 3 headings?
    |       +-- False --> VerdictNode(score=0)
    |       +-- True --> NonBinaryJudgementNode: Correct order?
    |                       +-- "Yes" --> VerdictNode(score=10)
    |                       +-- "Two out of order" --> VerdictNode(score=4)
    |                       +-- "All out of order" --> VerdictNode(score=2)
    +-- NonBinaryJudgementNode: (also depends on extracted headings)
```

In [ ]:
# Build the DAG from the docs example

# Step 1: Define the deepest node - ordering check (NonBinary)
correct_order_node = NonBinaryJudgementNode(
    criteria="Are the summary headings in the correct order: 'intro' => 'body' => 'conclusion'?",
    children=[
        VerdictNode(verdict="Yes", score=10),
        VerdictNode(verdict="Two are out of order", score=4),
        VerdictNode(verdict="All out of order", score=2),
    ],
)

# Step 2: Binary check - does it have all 3 headings?
correct_headings_node = BinaryJudgementNode(
    criteria="Does the summary headings contain all three: 'intro', 'body', and 'conclusion'?",
    children=[
        VerdictNode(verdict=False, score=0),
        VerdictNode(verdict=True, child=correct_order_node),  # If True, check order
    ],
)

# Step 3: TaskNode extracts headings from actual_output
extract_headings_node = TaskNode(
    instructions="Extract all headings in `actual_output`",
    evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT],
    output_label="Summary headings",
    children=[correct_headings_node, correct_order_node],
)

# Step 4: Create the DAG
dag = DeepAcyclicGraph(root_nodes=[extract_headings_node])

# Step 5: Create DAGMetric
format_correctness = DAGMetric(name="Format Correctness", dag=dag, model="gpt-4o-mini")

print("DAG built successfully with 4 nodes.")
print("Decision tree: Extract headings -> Check all present -> Check order -> Score")

In [ ]:
# Test the Format Correctness DAG with different cases

# Case 1: Perfect format (all headings, correct order)
perfect_case = LLMTestCase(
    input="Summarize the meeting.",
    actual_output="""Intro:
Alice outlined the agenda: product updates, blockers, and marketing alignment.

Body:
Bob reported performance fixes expected by Friday. Charlie requested finalized messaging by Monday. Bob confirmed an early stable build would be ready.

Conclusion:
The team aligned on next steps: engineering finalizing fixes, marketing preparing content, and a follow-up sync for Wednesday.""",
)

# Case 2: Wrong order (conclusion before body)
wrong_order_case = LLMTestCase(
    input="Summarize the meeting.",
    actual_output="""Intro:
The meeting covered product updates and marketing alignment.

Conclusion:
Everyone agreed on next steps and a Wednesday follow-up.

Body:
Bob discussed performance fixes. Charlie needs messaging by Monday.""",
)

# Case 3: Missing headings (no intro)
missing_heading_case = LLMTestCase(
    input="Summarize the meeting.",
    actual_output="""Body:
Bob reported on performance fixes. Charlie requested messaging by Monday.

Conclusion:
The team agreed on next steps.""",
)

# Run all three
for name, tc in [("Perfect format", perfect_case), ("Wrong order", wrong_order_case), ("Missing heading", missing_heading_case)]:
    format_correctness.measure(tc)
    print(f"{name:20s}: Score = {format_correctness.score:.4f} | Reason: {format_correctness.reason[:80]}...")

### DAG Example 2: RAG Answer Quality Evaluation

Here is a custom DAG designed for evaluating RAG pipeline answers. It checks:
1. Is the answer grounded in the retrieval context? (faithfulness check)
2. If grounded, is it complete? (coverage check)
3. If not grounded, is it a total hallucination or partially supported?

**Decision tree:**
```
TaskNode: Extract claims from actual_output
    +-- BinaryJudgementNode: Are ALL claims supported by retrieval_context?
            +-- True --> NonBinaryJudgementNode: How complete?
            |               +-- "Fully complete" --> score=10
            |               +-- "Partially complete" --> score=7
            |               +-- "Minimal coverage" --> score=4
            +-- False --> NonBinaryJudgementNode: How much unsupported?
                            +-- "Minor unsupported" --> score=5
                            +-- "Major unsupported" --> score=2
                            +-- "Completely fabricated" --> score=0
```

In [ ]:
# Build a RAG-specific DAG

# Completeness sub-tree (when answer IS grounded)
completeness_node = NonBinaryJudgementNode(
    criteria="How completely does the actual output answer the input question based on the retrieval context?",
    children=[
        VerdictNode(verdict="Fully complete - covers all key points", score=10),
        VerdictNode(verdict="Partially complete - covers some key points", score=7),
        VerdictNode(verdict="Minimal coverage - only touches on the topic", score=4),
    ],
)

# Hallucination severity sub-tree (when answer is NOT grounded)
hallucination_severity_node = NonBinaryJudgementNode(
    criteria="How much of the actual output is unsupported by the retrieval context?",
    children=[
        VerdictNode(verdict="Minor unsupported claims - mostly grounded with small additions", score=5),
        VerdictNode(verdict="Major unsupported claims - significant fabrication", score=2),
        VerdictNode(verdict="Completely fabricated - no connection to context", score=0),
    ],
)

# Main grounding check
grounding_check = BinaryJudgementNode(
    criteria="Are ALL factual claims in the actual output supported by the retrieval context?",
    evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT, LLMTestCaseParams.RETRIEVAL_CONTEXT],
    children=[
        VerdictNode(verdict=True, child=completeness_node),
        VerdictNode(verdict=False, child=hallucination_severity_node),
    ],
)

# Extract claims from the answer
extract_claims = TaskNode(
    instructions="Extract all factual claims made in the actual output. List each claim as a separate item.",
    evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT],
    output_label="Extracted claims",
    children=[grounding_check],
)

# Build the DAG
rag_dag = DeepAcyclicGraph(root_nodes=[extract_claims])
rag_quality_metric = DAGMetric(
    name="RAG Answer Quality",
    dag=rag_dag,
    model="gpt-4o-mini",
    threshold=0.7,
)

print("RAG Quality DAG built with 5 nodes.")

In [ ]:
# Test the RAG Quality DAG on our pipeline test cases
print("RAG Answer Quality DAG - Scores across test cases:\n")

rag_dag_scores = []
for i, tc in enumerate(test_cases):
    rag_quality_metric.measure(tc)
    rag_dag_scores.append(rag_quality_metric.score)
    print(f"Q{i+1}: {rag_quality_metric.score:.4f} | {tc.input[:55]}...")

print(f"\nAvg RAG Quality (DAG): {np.mean(rag_dag_scores):.4f}")
print(f"Min: {min(rag_dag_scores):.4f} | Max: {max(rag_dag_scores):.4f}")
print(f"Score spread: {max(rag_dag_scores) - min(rag_dag_scores):.4f}")

### DAG Example 3: VerdictNode with GEval Child

A `VerdictNode` can delegate scoring to a GEval metric instead of using a fixed score. This is useful when one branch needs subjective evaluation while others have deterministic scores.

In this example, if the answer is on-topic, we use GEval to score its quality. If off-topic, we immediately score 0.

In [ ]:
# GEval as a child of VerdictNode — hybrid DAG+GEval
quality_geval = GEval(
    name="Content Quality",
    criteria="Rate the quality of the actual output in terms of accuracy, helpfulness, and clarity.",
    evaluation_params=[LLMTestCaseParams.INPUT, LLMTestCaseParams.ACTUAL_OUTPUT],
    model="gpt-4o-mini",
)

# DAG: Check structure first, then evaluate quality if structure is OK
structure_check = BinaryJudgementNode(
    criteria="Does the actual output directly answer the question asked in the input (not off-topic or evasive)?",
    evaluation_params=[LLMTestCaseParams.INPUT, LLMTestCaseParams.ACTUAL_OUTPUT],
    children=[
        VerdictNode(verdict=True, child=quality_geval),   # Delegate to GEval
        VerdictNode(verdict=False, score=0),                # Immediate 0 if off-topic
    ],
)

hybrid_dag = DeepAcyclicGraph(root_nodes=[structure_check])
hybrid_metric = DAGMetric(name="Hybrid DAG+GEval", dag=hybrid_dag, model="gpt-4o-mini")

# Test: on-topic vs off-topic
on_topic = LLMTestCase(
    input="What is the return policy?",
    actual_output="Acme Corp offers a 30-day return policy. Items must be unused and in original packaging.",
)
off_topic = LLMTestCase(
    input="What is the return policy?",
    actual_output="The weather today is sunny with a high of 72 degrees. Perfect day for outdoor activities!",
)

hybrid_metric.measure(on_topic)
print(f"On-topic:  {hybrid_metric.score:.4f} | {hybrid_metric.reason[:80]}...")

hybrid_metric.measure(off_topic)
print(f"Off-topic: {hybrid_metric.score:.4f} | {hybrid_metric.reason[:80]}...")

### GEval vs DAG - Side-by-Side Comparison

Let's compare the same evaluation done with GEval vs DAG to see the difference in determinism and scoring behavior.

In [ ]:
# GEval approach: single prompt, LLM decides score
geval_format = GEval(
    name="Format Correctness (GEval)",
    evaluation_steps=[
        "The actual_output is completely wrong if it misses any of the headings: 'intro', 'body', 'conclusion'.",
        "If the actual_output has all the headings but in the wrong order, penalize it.",
        "If it has all correct headings in the right order, give a perfect score.",
    ],
    evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT],
    model="gpt-4o-mini",
)

# Run both on the same 3 test cases
cases = {
    "Perfect": perfect_case,
    "Wrong order": wrong_order_case,
    "Missing heading": missing_heading_case,
}

print(f"{'Case':<20s} {'GEval':>10s} {'DAG':>10s} {'Diff':>10s}")
print("-" * 52)

for name, tc in cases.items():
    geval_format.measure(tc)
    format_correctness.measure(tc)
    diff = abs(geval_format.score - format_correctness.score)
    print(f"{name:<20s} {geval_format.score:>10.4f} {format_correctness.score:>10.4f} {diff:>10.4f}")

print("\nDAG scores are deterministic (same every run), while GEval scores may vary.")

---
## Build Your Own DAG — Guided Exercise

Try building a DAG for **customer support quality** evaluation:

1. **Root node** (Binary): "Does the response address the customer's question?"
   - No → VerdictNode(score=0)
   - Yes → go to step 2
2. **Second node** (NonBinary): "What is the tone of the response?"
   - Professional → VerdictNode(score=1.0)
   - Neutral → VerdictNode(score=0.7)
   - Unprofessional → VerdictNode(score=0.2)

```python
# Your code here — define the nodes and build the DAG
# tone_professional = VerdictNode(verdict="Professional", score=1.0)
# ...
```

---
## Summary

### DAG Node Types
| Node | Purpose | Children |
|------|---------|----------|
| TaskNode | Preprocessing/extraction | 1 child |
| BinaryJudgementNode | Yes/No decision | 2 children (yes/no) |
| NonBinaryJudgementNode | Multiple-choice | N children |
| VerdictNode | Leaf with score | 0 or 1 GEval child |

### GEval vs DAG Decision Guide
| Scenario | Use GEval | Use DAG |
|----------|-----------|--------|
| Subjective quality assessment | ✓ | |
| Format/structure checking | | ✓ |
| Multi-step decision logic | | ✓ |
| Hybrid (rules + subjective) | | ✓ (with GEval child) |

### Next Steps
- **Notebook 07**: Datasets & synthetic test generation